# Comprehensive Sweep: Cross-Dataset Analysis**Date:** 2026-04-06 (updated from 2026-03-22)  **Datasets:** M4 (all 6 periods: Yearly, Quarterly, Monthly, Weekly, Daily, Hourly), Tourism-Yearly, Weather-96, Milk  **Configs:** 112 per dataset (M4-Daily: 14 only)  **Total runs:** ~9,521 (5,752 M4 + 1,120 Tourism + 1,529 Weather + 1,120 Milk)This notebook analyzes the comprehensive sweep of 112 N-BEATS configurations across 4 datasets (9 dataset-periods).**Key questions:**1. Which architectures generalize across datasets vs. which are dataset-specific winners?2. Do the 12 prior findings hold under controlled conditions?3. What is the Pareto frontier of quality vs. parameter count?4. How do optimal settings vary by dataset (active_g, backbone, depth, wavelet family)?5. Where are the biggest coverage gaps for future experiments?

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlibimport seaborn as snsfrom scipy import statsimport warningswarnings.filterwarnings('ignore')matplotlib.rcParams['figure.figsize'] = (14, 6)matplotlib.rcParams['font.size'] = 11sns.set_style('whitegrid')# Load all datasetsm4_all = pd.read_csv('../../results/m4/comprehensive_sweep_m4_results.csv')tourism = pd.read_csv('../../results/tourism/comprehensive_sweep_tourism_results.csv')weather = pd.read_csv('../../results/weather/comprehensive_sweep_weather_results.csv')milk = pd.read_csv('../../results/milk/comprehensive_sweep_milk_results.csv')# Split M4 into periodsm4y = m4_all[m4_all['period'] == 'Yearly'].copy()m4q = m4_all[m4_all['period'] == 'Quarterly'].copy()m4m = m4_all[m4_all['period'] == 'Monthly'].copy()m4w = m4_all[m4_all['period'] == 'Weekly'].copy()m4d = m4_all[m4_all['period'] == 'Daily'].copy()m4h = m4_all[m4_all['period'] == 'Hourly'].copy()datasets = {    'M4-Yearly': m4y, 'M4-Quarterly': m4q, 'M4-Monthly': m4m,    'M4-Weekly': m4w, 'M4-Daily': m4d, 'M4-Hourly': m4h,    'Tourism': tourism, 'Weather': weather, 'Milk': milk}# Summaryfor name, df in datasets.items():    nd = df[df['diverged']==False]    metric = 'mse' if name == 'Weather' else 'smape'    print(f"{name:15s}: {len(df):5d} rows, {df['config_name'].nunique():3d} configs, "          f"{df['diverged'].sum():3d} diverged ({df['diverged'].mean():.1%}), "          f"SMAPE range [{nd['smape'].min():.2f}, {nd['smape'].max():.2f}]")

In [ ]:
# Helper: compute per-config summary for a datasetdef config_summary(df, metric='smape'):    d = df[df['diverged']==False]    grp = d.groupby('config_name').agg(        mean=(metric, 'mean'), std=(metric, 'std'), median=(metric, 'median'),        n=(metric, 'count'), n_params=('n_params', 'first'),        backbone=('backbone', 'first'), category=('category', 'first'),        active_g_cfg=('active_g_cfg', 'first'), arch_family=('arch_family', 'first'),        wavelet_family=('wavelet_family', 'first'), basis_dim_label=('basis_dim_label', 'first'),        latent_dim_cfg=('latent_dim_cfg', 'first'), trend_thetas_dim_cfg=('trend_thetas_dim_cfg', 'first'),        skip_distance_cfg=('skip_distance_cfg', 'first'), block_type_primary=('block_type_primary', 'first'),        stack_pattern=('stack_pattern', 'first'), n_stacks=('n_stacks', 'first'),    ).reset_index()    # Add divergence rate from full data    div = df.groupby('config_name')['diverged'].mean().reset_index()    div.columns = ['config_name', 'div_rate']    grp = grp.merge(div, on='config_name', how='left')    return grp.sort_values('mean')summaries = {name: config_summary(df) for name, df in datasets.items()}# Weather also with MSEsummaries_mse = {'Weather': config_summary(weather, metric='mse')}print("Summaries computed for all 9 dataset-periods.")

## 1. Cross-Dataset Ranking: Who Generalizes?The most important question: which architectures perform well *everywhere*, not just on one dataset?We rank each config 1-112 per dataset by mean SMAPE, then compute the average rank.

In [ ]:
# Cross-dataset ranking (core datasets: M4-Y, M4-Q, Tourism, Weather, Milk)core_datasets = ['M4-Yearly', 'M4-Quarterly', 'Tourism', 'Weather', 'Milk']rank_cols = {}for name in core_datasets:    s = summaries[name]    s_ranked = s[['config_name', 'mean']].copy()    s_ranked['rank'] = s_ranked['mean'].rank()    rank_cols[name] = s_ranked.set_index('config_name')['rank']rank_df = pd.DataFrame(rank_cols)rank_df['mean_rank'] = rank_df.mean(axis=1)rank_df['n_datasets'] = rank_df.notna().sum(axis=1)rank_df['worst_rank'] = rank_df[core_datasets].max(axis=1)rank_df['best_rank'] = rank_df[core_datasets].min(axis=1)rank_df = rank_df[rank_df['n_datasets'] >= 4].sort_values('mean_rank')# Add metadatameta = summaries['M4-Yearly'][['config_name','n_params','backbone','category']].set_index('config_name')rank_df = rank_df.join(meta)print("TOP 25 CROSS-DATASET CONFIGS (by mean rank across 5 core datasets)")print("="*140)print(f"{'Rank':>4s}  {'Config':<45s} {'MeanR':>6s} {'M4Y':>4s} {'M4Q':>4s} {'Tour':>5s} {'Wx':>4s} {'Milk':>5s} {'Worst':>5s} {'Params':>8s} {'Category'}")print("-"*140)for i, (cn, row) in enumerate(rank_df.head(25).iterrows(), 1):    r = [f"{row[d]:.0f}" if pd.notna(row[d]) else "-" for d in core_datasets]    params = f"{row['n_params']/1000:.0f}K" if pd.notna(row.get('n_params')) else "?"    cat = row.get('category', '?')    print(f"{i:4d}  {cn:<45s} {row['mean_rank']:6.1f} {r[0]:>4s} {r[1]:>4s} {r[2]:>5s} {r[3]:>4s} {r[4]:>5s} {row['worst_rank']:5.0f} {params:>8s} {cat}")

In [ ]:
# Visualize: heatmap of ranks for top 20 configsfig, ax = plt.subplots(figsize=(12, 10))top20 = rank_df.head(20)[core_datasets]short = {n: n.replace('TALG+', 'T+').replace('V3ALG', 'V3A') for n in top20.index}top20.index = [short.get(n, n) for n in top20.index]sns.heatmap(top20, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax,            linewidths=0.5, vmin=1, vmax=112, cbar_kws={'label': 'Rank (1=best)'})ax.set_title('Cross-Dataset Rank Heatmap (Top 20 by Mean Rank)', fontsize=14)ax.set_xlabel('')plt.tight_layout()plt.show()print("\nKey insight: No single config ranks top-20 on ALL datasets.")print("TALG+DB3V3ALG_10s_ag0 is the best generalist (mean rank 14.6).")print("Dataset-specific tuning is essential for optimal performance.")

## 2. Category-Level PerformanceArchitecture families compared across all 9 dataset-periods. Green = best, red = worst per dataset.

In [ ]:
# Category-level ranking per dataset (3x3 grid for 9 dataset-periods)fig, axes = plt.subplots(3, 3, figsize=(22, 16))axes = axes.flatten()for idx, (ds_name, df) in enumerate(datasets.items()):    ax = axes[idx]    d = df[df['diverged']==False]    cat_means = d.groupby('category')['smape'].mean().sort_values()    colors = ['#2ecc71' if v == cat_means.min() else '#e74c3c' if v == cat_means.max() else '#3498db' for v in cat_means.values]    cat_means.plot.barh(ax=ax, color=colors)    ax.set_title(f'{ds_name}', fontsize=12)    ax.set_xlabel('Mean SMAPE')    ax.invert_yaxis()plt.suptitle('Mean SMAPE by Architecture Category (lower = better)', fontsize=15, y=1.01)plt.tight_layout()plt.show()print("Key patterns:")print("- TrendWavelet variants consistently near top on M4-Y/Q/M, Tourism, Milk")print("- Alternating stacks (alt_trend_wavelet_rb, alt_aelg) excel on Weather and M4-Weekly")print("- Paper baselines dominate M4-Daily and M4-Hourly (long horizons)")print("- BottleneckGeneric/GenericAELG variants are reliably bottom-tier")

## 3. M4 All-Period AnalysisThe M4 dataset spans 6 periods with horizons from H=6 (Yearly) to H=48 (Hourly). Architecture preferences shift with horizon length.

In [ ]:
# M4: Best config per periodprint("M4 WINNERS BY PERIOD")print("="*120)m4_periods = ['M4-Yearly', 'M4-Quarterly', 'M4-Monthly', 'M4-Weekly', 'M4-Daily', 'M4-Hourly']horizons = {'M4-Yearly': 6, 'M4-Quarterly': 8, 'M4-Monthly': 18, 'M4-Weekly': 13, 'M4-Daily': 14, 'M4-Hourly': 48}for ds_name in m4_periods:    s = summaries[ds_name]    best = s.iloc[0]    # Best baseline    baselines = s[s['category'] == 'paper_baseline']    best_bl = baselines.iloc[0] if len(baselines) > 0 else None        print(f"\n{ds_name} (H={horizons[ds_name]}):")    print(f"  Winner: {best['config_name']:<45s} SMAPE={best['mean']:.3f} +/-{best['std']:.3f}  Params={best['n_params']/1000:.0f}K  [{best['category']}]")    if best_bl is not None:        delta = (best['mean'] - best_bl['mean']) / best_bl['mean'] * 100        print(f"  Best baseline: {best_bl['config_name']:<40s} SMAPE={best_bl['mean']:.3f}  Delta: {delta:+.2f}%")        # Best sub-1M config    small = s[s['n_params'] < 1_000_000]    if len(small) > 0:        best_sm = small.iloc[0]        delta_sm = (best_sm['mean'] - best['mean']) / best['mean'] * 100        print(f"  Best <1M:  {best_sm['config_name']:<40s} SMAPE={best_sm['mean']:.3f}  Delta vs winner: {delta_sm:+.2f}%  Params={best_sm['n_params']/1000:.0f}K")

In [ ]:
# Heatmap: category performance across M4 periodsm4_cat_means = {}for ds_name in m4_periods:    d = datasets[ds_name][datasets[ds_name]['diverged']==False]    m4_cat_means[ds_name] = d.groupby('category')['smape'].mean()cat_df = pd.DataFrame(m4_cat_means)# Rank within each periodcat_ranks = cat_df.rank()fig, ax = plt.subplots(figsize=(14, 8))sns.heatmap(cat_ranks, annot=True, fmt='.0f', cmap='RdYlGn_r', ax=ax,            linewidths=0.5, cbar_kws={'label': 'Category Rank (1=best)'})ax.set_title('Architecture Category Ranks Across M4 Periods', fontsize=14)plt.tight_layout()plt.show()print("\nHorizon-dependent preferences:")print("- Short horizons (H=6-8): Unified TrendWavelet and TWAELG dominate")print("- Medium horizons (H=13-18): Mixed; alternating stacks become competitive")print("- Long horizons (H=14-48): Paper baselines and deep stacks preferred")print("- Daily (H=14) only tested 14 configs -- needs more coverage!")

In [ ]:
# Stack depth: 10 vs 30 across all datasetsprint("STACK DEPTH ANALYSIS: 10 vs 30 stacks")print("="*100)for ds_name in ['M4-Yearly', 'M4-Quarterly', 'M4-Monthly', 'M4-Weekly', 'M4-Hourly', 'Tourism', 'Weather', 'Milk']:    s = summaries[ds_name]    s10 = s[s['n_stacks'] == 10]    s30 = s[s['n_stacks'] == 30]        if len(s10) > 0 and len(s30) > 0:        # Match configs that have both 10 and 30 variants        configs_10 = set(s10['config_name'].str.replace('_10s_', '_Xs_').str.replace('10s', 'Xs'))        configs_30 = set(s30['config_name'].str.replace('_30s_', '_Xs_').str.replace('30s', 'Xs'))                mean_10 = s10['mean'].mean()        mean_30 = s30['mean'].mean()        winner = "10s" if mean_10 < mean_30 else "30s"        print(f"  {ds_name:15s}: 10s mean={mean_10:.3f} ({len(s10)} configs)  30s mean={mean_30:.3f} ({len(s30)} configs)  Winner: {winner}")

## 4. Hyperparameter Sensitivity### active_g Effect (Dataset-Level, NOT Block-Level)The most important revision: `active_g=forecast` is a **dataset-level** setting. It is:- **Essential** on Tourism (p=0.0002) and for Generic blocks on Milk- **Catastrophic** on Weather for unified/homogeneous stacks (SMAPE ~100)- **Benign/beneficial** on Weather for alternating stacks only- **Mixed** on M4 (helps Yearly/Hourly, hurts Monthly)

In [ ]:
# active_g effect across datasetsprint("ACTIVE_G EFFECT: ag0 vs agf")print("="*100)for ds_name in ['M4-Yearly', 'M4-Quarterly', 'M4-Monthly', 'M4-Weekly', 'M4-Hourly', 'Tourism', 'Weather', 'Milk']:    s = summaries[ds_name]    ag0 = s[s['active_g_cfg'] == 'False']    agf = s[s['active_g_cfg'] == 'forecast']        if len(ag0) > 0 and len(agf) > 0:        mean_ag0 = ag0['mean'].mean()        mean_agf = agf['mean'].mean()        delta = mean_agf - mean_ag0        print(f"  {ds_name:15s}: ag0 mean={mean_ag0:.3f} ({len(ag0)} configs)  agf mean={mean_agf:.3f} ({len(agf)} configs)  Delta: {delta:+.3f}")# Weather: nuanced view by stack patternprint("\nWEATHER active_g BY STACK PATTERN (nuanced finding):")print("-"*80)wx = weather[weather['diverged']==False]for pattern in wx['stack_pattern'].unique():    subset = wx[wx['stack_pattern'] == pattern]    for ag in ['False', 'forecast']:        ag_sub = subset[subset['active_g_cfg'] == ag]        if len(ag_sub) > 0:            print(f"  {pattern:20s} ag={ag:8s}: mean SMAPE={ag_sub['smape'].mean():.2f} (n={len(ag_sub)})")print("\n=> active_g=forecast is catastrophic for homogeneous/unified stacks on Weather")print("   but benign or BENEFICIAL for alternating/hybrid stacks!")

In [ ]:
# Weather: ag0 vs agf by stack patternfig, axes = plt.subplots(1, 2, figsize=(16, 6))# Left: overall histogramd_wx = weather[weather['diverged']==False]ag0_means = d_wx[d_wx['active_g_cfg']=='False'].groupby('config_name')['smape'].mean()agf_means = d_wx[d_wx['active_g_cfg']=='forecast'].groupby('config_name')['smape'].mean()axes[0].hist(ag0_means, bins=20, alpha=0.7, label=f'ag0 (n={len(ag0_means)})', color='#3498db')axes[0].hist(agf_means, bins=20, alpha=0.7, label=f'agf (n={len(agf_means)})', color='#e74c3c')axes[0].axvline(x=50, color='black', linestyle='--', alpha=0.5, label='Catastrophe threshold')axes[0].set_xlabel('Mean SMAPE per config')axes[0].set_ylabel('Count')axes[0].set_title('Weather-96: active_g Overall Distribution')axes[0].legend()# Right: by stack pattern  agf_by_pattern = d_wx[d_wx['active_g_cfg']=='forecast'].groupby(['config_name','stack_pattern'])['smape'].mean().reset_index()for pat, color in [('alternating', '#2ecc71'), ('homogeneous', '#e74c3c'), ('prefix_body', '#f39c12')]:    sub = agf_by_pattern[agf_by_pattern['stack_pattern']==pat]    if len(sub) > 0:        axes[1].hist(sub['smape'], bins=15, alpha=0.6, label=f'{pat} (n={len(sub)})', color=color)axes[1].set_xlabel('Mean SMAPE (agf configs only)')axes[1].set_title('Weather-96: agf by Stack Pattern')axes[1].legend()plt.tight_layout()plt.show()

### Wavelet Family AnalysisWavelet choice is significant on 3/4 datasets (KW p<0.05). The optimal wavelet is dataset-dependent, but **db3 is the safest cross-dataset default**.

In [ ]:
# Wavelet family analysis per datasetprint("WAVELET FAMILY ANALYSIS")print("="*100)for ds_name in ['M4-Yearly', 'M4-Monthly', 'M4-Hourly', 'Tourism', 'Weather', 'Milk']:    s = summaries[ds_name]    wav = s[s['wavelet_family'].notna() & (s['wavelet_family'] != 'none')]    if len(wav) == 0:        continue        print(f"\n{ds_name}:")    fam_means = wav.groupby('wavelet_family')['mean'].agg(['mean','std','count']).sort_values('mean')    for fam, row in fam_means.iterrows():        print(f"  {fam:10s}: SMAPE={row['mean']:.3f} +/-{row['std']:.3f} (n={row['count']:.0f})")        # KW test    groups = [wav[wav['wavelet_family']==f]['mean'].values for f in wav['wavelet_family'].unique()]    groups = [g for g in groups if len(g) >= 2]    if len(groups) >= 2:        h, p = stats.kruskal(*groups)        print(f"  Kruskal-Wallis: H={h:.2f}, p={p:.4f} {'***' if p<0.01 else '**' if p<0.05 else '*' if p<0.1 else 'ns'}")

### Backbone Hierarchy (Dataset-Dependent!)The backbone ranking **reverses between M4/Tourism and Weather/Milk:**- M4/Tourism: RootBlock > AELG > AE- Weather: AE > AELG > RootBlock - Milk: AE > AELG

In [ ]:
# Backbone hierarchy per datasetprint("BACKBONE HIERARCHY")print("="*100)for ds_name in ['M4-Yearly', 'M4-Quarterly', 'Tourism', 'Weather', 'Milk']:    s = summaries[ds_name]    bb_means = s.groupby('backbone')['mean'].agg(['mean','median','count']).sort_values('mean')    print(f"\n{ds_name}:")    for bb, row in bb_means.iterrows():        print(f"  {bb:20s}: mean={row['mean']:.3f}  median={row['median']:.3f}  (n={row['count']:.0f})")        # Divergence by backbone (for datasets with divergence)    div_df = s.groupby('backbone')['div_rate'].mean()    if div_df.max() > 0:        print(f"  Divergence: {dict(zip(div_df.index, [f'{v:.1%}' for v in div_df.values]))}")

In [ ]:
# Latent dimension analysisprint("LATENT DIMENSION ANALYSIS")print("="*100)for ds_name in ['M4-Yearly', 'Tourism', 'Weather', 'Milk']:    s = summaries[ds_name]    ld = s[s['latent_dim_cfg'].notna() & (s['latent_dim_cfg'] != 'none')]    if len(ld) == 0:        continue        # Convert to numeric    ld = ld.copy()    ld['ld_num'] = pd.to_numeric(ld['latent_dim_cfg'], errors='coerce')    ld = ld[ld['ld_num'].notna()]        print(f"\n{ds_name}:")    ld_means = ld.groupby('ld_num')['mean'].agg(['mean','std','count']).sort_values('mean')    for ldv, row in ld_means.iterrows():        print(f"  ld={int(ldv):3d}: SMAPE={row['mean']:.3f} +/-{row['std']:.3f} (n={row['count']:.0f})")        groups = [ld[ld['ld_num']==v]['mean'].values for v in ld['ld_num'].unique()]    groups = [g for g in groups if len(g) >= 2]    if len(groups) >= 2:        h, p = stats.kruskal(*groups)        print(f"  KW: H={h:.2f}, p={p:.4f}")

## 5. Pareto Frontier: Quality vs. ParametersThe most actionable view: which configs give the best bang per parameter? Sub-1M models are within 0.5% of winners on most datasets.

In [ ]:
# Pareto frontier plots (2x2: M4-Y, M4-Monthly, Weather, Milk)fig, axes = plt.subplots(2, 2, figsize=(18, 14))axes = axes.flatten()pareto_datasets = ['M4-Yearly', 'M4-Monthly', 'Weather', 'Milk']for ax, ds_name in zip(axes, pareto_datasets):    s = summaries[ds_name]        colors = {'RootBlock': '#3498db', 'AERootBlockLG': '#e74c3c', 'AERootBlock': '#2ecc71', 'AERootBlockVAE': '#9b59b6'}    for bb, color in colors.items():        mask = s['backbone'] == bb        if mask.any():            ax.scatter(s[mask]['n_params']/1e6, s[mask]['mean'], c=color, alpha=0.6, s=50, label=bb)        # Highlight Pareto frontier    pareto = []    for _, row in s.iterrows():        dominated = ((s['mean'] <= row['mean']) & (s['n_params'] <= row['n_params']) &                     ((s['mean'] < row['mean']) | (s['n_params'] < row['n_params']))).any()        if not dominated:            pareto.append(row)    if pareto:        pareto_df = pd.DataFrame(pareto).sort_values('n_params')        ax.plot(pareto_df['n_params']/1e6, pareto_df['mean'], 'k--', alpha=0.5, linewidth=2)        for _, row in pareto_df.iterrows():            short_name = row.config_name[:25]            ax.annotate(short_name, (row.n_params/1e6, row['mean']), fontsize=6,                       xytext=(5, 5), textcoords='offset points')        ax.set_xlabel('Parameters (millions)')    ax.set_ylabel('Mean SMAPE')    ax.set_title(f'{ds_name}: Quality vs Parameters')    ax.legend(fontsize=8)    ax.set_xscale('log')plt.tight_layout()plt.show()print("The ~400K param regime (TWAELG, TWAE, TWGAELG) consistently sits on the Pareto frontier.")print("Increasing params beyond ~3M gives diminishing or zero returns on most datasets.")print("Paper baselines at 8-50M params are massively over-parameterized.")

## 6. Dataset Winners and Overall Recommendations

In [ ]:
# Final summary: Winners per datasetprint("DATASET WINNERS (Comprehensive Sweep)")print("="*120)winners = {    'M4-Yearly (H=6)':    ('TW_10s_td3_bdeq_coif2',       13.499, 2076,  'Unified TrendWavelet RB'),    'M4-Quarterly (H=8)': ('NBEATS-IG_10s_ag0',            10.126, 19644, 'Paper baseline (15-way tie)'),    'M4-Monthly (H=18)':  ('TW_30s_td3_bd2eq_coif2',       13.279, 7100,  'Unified TrendWavelet RB'),    'M4-Weekly (H=13)':   ('T+Db3V3_30s_bdeq',             6.671,  15800, 'Alt Trend+WaveletV3'),    'M4-Daily (H=14)':    ('NBEATS-G_30s_ag0',             2.603,  26000, 'Paper baseline (14 configs only!)'),    'M4-Hourly (H=48)':   ('NBEATS-IG_30s_agf',            8.587,  43600, 'Paper baseline'),    'Tourism-Y (H=4)':    ('TW_10s_td3_bdeq_db3',          21.773, 2000,  'Unified TrendWavelet RB'),    'Weather-96':          ('TAE+DB3V3AE_30s_ld8_ag0',      41.640, 7100,  'Alt TrendAE+WaveletV3AE (MSE=0.138)'),    'Milk (H=6)':          ('TALG+DB3V3ALG_10s_ag0',        1.512,  1000,  'Alt AELG (reliable; rank 1 has 50% div)'),}for ds, (cn, smape, params, desc) in winners.items():    print(f"  {ds:22s}: {cn:<40s} SMAPE={smape:.3f} | {params/1000:.0f}K | {desc}")print("\n" + "="*120)print("BEST GENERALIST: TALG+DB3V3ALG_10s_ag0")print("  Mean rank = 14.6 across 5 core datasets, 2,390K params")print("  Uses ag0 (safe on ALL datasets including Weather)")print("\nBEST PARETO-OPTIMAL: TWAELG_10s_ld16_coif2_ag0 (436K params)")print("  Within 0.5% of winner on M4-Y/Q/M/W, Tourism, Milk")print("  20-50x fewer params than baselines")print("\nKEY INSIGHT: Architecture preferences shift with horizon length:")print("  Short (H=4-8):  Unified TrendWavelet / TWAELG, 10 stacks")print("  Medium (H=13-18): Mixed; alternating becomes competitive")print("  Long (H=48+):   Paper baselines or alternating AE, 30 stacks")

## 7. Bimodal Convergence and DivergenceDivergence rates vary dramatically by dataset. The AE bottleneck is the primary stabilization mechanism.

In [ ]:
# Divergence analysisfig, axes = plt.subplots(1, 3, figsize=(18, 6))# Milk divergence by categorymilk_div = milk.groupby('category')['diverged'].mean().sort_values(ascending=False)milk_div.plot.barh(ax=axes[0], color=['#e74c3c' if v > 0.1 else '#f39c12' if v > 0.01 else '#2ecc71' for v in milk_div.values])axes[0].set_title('Milk: Divergence Rate by Category')axes[0].set_xlabel('Divergence Rate')# Milk divergence by backbonemilk_bb_div = milk.groupby('backbone')['diverged'].mean().sort_values(ascending=False)milk_bb_div.plot.bar(ax=axes[1], color=['#e74c3c' if v > 0.1 else '#2ecc71' for v in milk_bb_div.values])axes[1].set_title('Milk: Divergence by Backbone')axes[1].set_ylabel('Divergence Rate')axes[1].tick_params(axis='x', rotation=45)# Tourism bimodal detectiontour_configs = tourism.groupby('config_name')['smape'].agg(['mean','std','min','max'])tour_configs['range'] = tour_configs['max'] - tour_configs['min']bimodal = tour_configs[tour_configs['range'] > 30].sort_values('range', ascending=False)if len(bimodal) > 0:    bimodal['range'].head(10).plot.barh(ax=axes[2], color='#e74c3c')    axes[2].set_title(f'Tourism: Bimodal Configs (range>30)')    axes[2].set_xlabel('SMAPE Range (max - min)')else:    axes[2].text(0.5, 0.5, 'No bimodal configs detected', ha='center', va='center', transform=axes[2].transAxes)    axes[2].set_title('Tourism: Bimodal Detection')plt.tight_layout()plt.show()print("Divergence summary:")for ds_name in ['M4-Yearly', 'Tourism', 'Weather', 'Milk']:    df = datasets[ds_name]    rate = df['diverged'].mean()    print(f"  {ds_name:15s}: {rate:.1%} ({df['diverged'].sum()}/{len(df)})")print("\nMitigation: AE bottleneck (1.7% vs 40.6% on Milk), active_g=forecast (Tourism), TrendWavelet blocks (immune)")

## 8. Finding Verification Summary (F1-F12)| # | Finding | Verdict | Key Evidence ||---|---------|---------|-------------|| F1 | active_g stabilizes Generic, hurts TrendWavelet | **REVISED** | Dataset-level AND architecture-level. Catastrophic on Weather unified; essential on Tourism; critical for Generic on Milk. Alternating stacks tolerate it on Weather. || F2 | trend_thetas_dim=3 is best | **PARTIALLY CONFIRMED** | td5 beats td3 on M4-Yearly (p=0.009). td3 wins on Tourism/Milk. Safe default. || F3 | Wavelet type barely matters | **DENIED** | Significant on 3/4 datasets (KW p<0.05). Best wavelet is dataset-dependent. db3 safest cross-dataset. || F4 | basis_dim=eq_fcast is best | **PARTIALLY CONFIRMED** | bdeq for short horizons, bd2eq for longer. Neither dominant. || F5 | Higher ld hurts AE, helps AELG | **DENIED** | No consistent pattern. Weather reverses (ld=8 > ld=16 > ld=32). || F6 | Skip connections rescue deep stacks | **DENIED** | Hurts on M4-Y, Tourism, Milk. Only marginal benefit on Weather. || F7 | active_g rescues Generic without skip | **DATASET-DEPENDENT** | True on Milk (massive). Catastrophic on Weather unified. || F8 | Backbone: AELG >= RB > AE | **DATASET-DEPENDENT** | M4/Tourism: RB > AELG > AE. Weather/Milk: AE > AELG > RB (reversed!). || F9 | Novel arches match baselines at fewer params | **CONFIRMED** | 6/9 dataset-periods. Weather: 25% better MSE at 3.6x fewer params. Strongest finding. || F10 | Alt AELG = top quality | **PARTIALLY CONFIRMED** | #1 Weather, #2 Milk. Weak on Tourism. Not universal. || F11 | TrendAE+WaveletV3AE = most stable | **CONFIRMED (Milk/Weather)** | AE: 1.7% divergence vs RB: 40.6% on Milk. Zero divergence on Weather. || F12 | TWAELG/TWAE ~400K = Pareto optimal | **CONFIRMED** | On Pareto frontier of ALL datasets. Strongest confirmed finding. |**Bottom line:** Only 2 of 12 findings cleanly confirmed (F9, F12). Most "findings" from single-dataset studies do not generalize. The key revision is that **active_g, backbone hierarchy, and latent dim** are all dataset-dependent, not universal.

## 9. Priority Next Experiments1. **M4-Daily with wavelet architectures** -- Only 14 configs tested (baselines only). Biggest coverage gap.2. **Tourism: coif3 + eq_bcast** -- Known SOTA (20.864) uses settings not in this sweep. Could set new SOTA.3. **M4-Hourly with larger AE/AELG** -- TAE+DB3V3AE already #3 on Hourly. 30-stack AELG with agf could win.4. **Weather: TAE+DB3V3AE at 10s/20s** -- Find depth sweet spot between 30s winner and param efficiency.5. **Milk: LR warmup for non-AE stacks** -- T+Sym10V3_30s achieves SMAPE 1.262 but 50% diverge.6. **Cross-dataset generalist validation** -- T+HaarV3_30s_bd2eq on Tourism/Weather/Milk.**See full report:** `experiments/analysis/analysis_reports/comprehensive_sweep_cross_dataset_analysis.md`